# 02 流病視覺化：5 張關鍵圖表

用松柏護理之家退伍軍人症 line list，學會 matplotlib / seaborn / plotly 三大繪圖套件。

| 圖表 | 使用套件 | 觀察重點 |
|------|---------|----------|
| 流行曲線 | matplotlib | 傳播模式（共同暴露源 vs 持續傳播） |
| 年齡分布 | seaborn | 年齡是否為危險因子 |
| 翼區侵襲率 | seaborn | 空間聚集線索 |
| 嚴重度×共病 | seaborn heatmap | 多因子交互 |
| 互動分層曲線 | plotly | 各樓層流行高峰比較 |

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- 資料準備（與前一堂課相同） ---
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import plotly.express as px
import plotly.io as pio

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

# Plotly: 確保在靜態建置（jupyter-book build）時也能輸出互動圖
pio.renderers.default = "notebook"

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# 日期轉換
date_cols = [
    "facility_admission_date", "symptom_onset_date",
    "hospitalization_date", "death_date", "notification_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# 衍生變項
comorbidity_cols = [
    "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "comorbidity_copd", "immunosuppressed",
]
df["n_comorbidities"] = df[comorbidity_cols].sum(axis=1)
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["age_group"] = pd.cut(
    df["age"], bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

# 只取感染者（畫流行曲線等用）
cases = df[df["infected"] == 1].copy()
print(f"全體：{len(df)} 人，感染者：{len(cases)} 人")

## 1) 流行曲線 — Epidemic Curve（matplotlib）

最經典的流病圖表。X 軸是發病日，Y 軸是每日新增病例。
曲線形狀可推斷傳播模式：尖峰 → 共同暴露源；拖尾 → 持續傳播。

### 繪製要點（CDC / ECDC 規範）

流行曲線本質上是**直方圖（histogram）**，不是一般的長條圖（bar chart）：

- **相鄰長條不留間隙**：X 軸是連續時間軸，長條之間不應有空隙（`width=1.0`）
- **補齊沒有病例的日期**：即使某天 0 例也要佔位（用 `reindex` 填 0），否則 X 軸間距失真
- **顯示爆發前背景期**：包含疫情爆發前 1–2 個潛伏期的日期，讓讀者看到疫情何時偏離背景值
- **標題要能獨立閱讀**：包含疾病名稱、地點、時間範圍
- **X 軸**：標示「發病日期（Date of Symptom Onset）」——明確說明時間基準
- **Y 軸**：標示「病例數（Number of Cases）」——必須是整數刻度，從 0 開始，不截斷
- **隱藏格線**：減少視覺干擾，去除上方和右方邊框
- **個案分類用顏色區分**：確診 vs 疑似須用不同顏色並附圖例
- **不在長條上標數字**：避免數位與類比資訊互相干擾

In [ ]:
import matplotlib.dates as mdates

daily = cases.groupby("symptom_onset_date").size().rename("cases")

# 補齊完整日期範圍：包含爆發前 3 天（顯示背景期）
date_range = pd.date_range(
    daily.index.min() - pd.Timedelta(days=3),
    daily.index.max() + pd.Timedelta(days=1),
    freq="D",
)
daily = daily.reindex(date_range, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    daily.index, daily.values,
    width=1.0,                         # 相鄰長條緊密貼合（直方圖風格）
    color="#2c7fb8", edgecolor="white", linewidth=0.5,
)
ax.set_title(
    "松柏護理之家退伍軍人症流行曲線，依發病日，2026 年 1 月",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("發病日期（Date of Symptom Onset）")
ax.set_ylabel("病例數（Number of Cases）")

# 日期格式化
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45)

# X 軸緊貼資料範圍、Y 軸從 0 開始且整數刻度
ax.set_xlim(
    daily.index.min() - pd.Timedelta(hours=12),
    daily.index.max() + pd.Timedelta(hours=12),
)
ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))  # Y 軸整數刻度

# CDC 風格：隱藏格線、去除上右邊框
ax.grid(False)
ax.spines["top"].set_visible(False)     # 去除上邊框
ax.spines["right"].set_visible(False)   # 去除右邊框
plt.tight_layout()
plt.show()

### 經典方格式流行曲線（Unit Chart）

教科書和 CDC 疫調報告中常見的方格式（stacked squares）流行曲線——每個小方格代表一個病例，堆疊形成柱狀。這裡我們用顏色區分確診與疑似個案。

> 💡 方格式特別適合**小規模群聚**（數十至百餘例）。病例數太大時方格會太小，改用標準直方圖更合適。

In [ ]:
# 準備每日 confirmed / probable 的病例數
daily_class = (
    cases.groupby(["symptom_onset_date", "case_classification"])
    .size()
    .unstack(fill_value=0)
)
daily_class = daily_class.reindex(date_range, fill_value=0)
colors_map = {"confirmed": "#2c7fb8", "probable": "#a6bddb"}

fig, ax = plt.subplots(figsize=(10, 5))
box_size = 1.0

for date in daily_class.index:
    x = mdates.date2num(date)
    j = 0  # 目前堆疊高度
    for cls in ["confirmed", "probable"]:
        count = daily_class.at[date, cls] if cls in daily_class.columns else 0
        for _ in range(int(count)):
            rect = plt.Rectangle(
                (x - box_size / 2, j * box_size),
                box_size, box_size,
                facecolor=colors_map[cls],
                edgecolor="white", linewidth=0.8,
            )
            ax.add_patch(rect)
            j += 1

# 座標軸設定
ax.set_xlim(
    mdates.date2num(daily_class.index.min()) - 1.5,
    mdates.date2num(daily_class.index.max()) + 1.5,
)
y_max = daily_class.sum(axis=1).max()
ax.set_ylim(0, y_max + 1)
ax.set_aspect("equal")

ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

ax.set_title(
    "松柏護理之家退伍軍人症流行曲線 — 方格式（依個案分類）",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("發病日期（Date of Symptom Onset）")
ax.set_ylabel("病例數（Number of Cases）")

# 手動圖例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2c7fb8", edgecolor="white", label="確診（Confirmed）"),
    Patch(facecolor="#a6bddb", edgecolor="white", label="疑似（Probable）"),
]
ax.legend(handles=legend_elements, loc="upper left", frameon=False)

ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 2) 年齡分布：感染 vs 未感染（seaborn）

把全體 280 人的年齡分布疊起來，看感染者是否集中在特定年齡層。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(
    data=df, x="age", hue="infected", bins=15,
    multiple="stack", palette={0: "#cccccc", 1: "#e34a33"}, ax=ax,
)
ax.set_title("年齡分布：感染 vs 未感染")
ax.set_xlabel("年齡")
ax.set_ylabel("人數")
ax.legend(title="感染", labels=["未感染", "感染"])
plt.tight_layout()
plt.show()

## 3) 各翼區侵襲率長條圖（seaborn）

不能直接比病例數——要除以分母（住民數）才公平。
侵襲率異常偏高的翼區可能有共同暴露源（例如淋浴設備）。

In [ ]:
# 計算翼區統計
wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(residents=("case_id", "size"), infected=("infected", "sum"))
    .reset_index()
)
wing_stats["attack_rate_pct"] = (
    wing_stats["infected"] / wing_stats["residents"] * 100
).round(1)
wing_stats["label"] = wing_stats["floor"].astype(str) + wing_stats["wing"]
wing_stats = wing_stats.sort_values("attack_rate_pct", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=wing_stats, x="label", y="attack_rate_pct",
    hue="label", palette="YlOrRd", legend=False, ax=ax,
)
ax.set_title("各翼區侵襲率比較")
ax.set_xlabel("翼區")
ax.set_ylabel("侵襲率 (%)")

# 在長條上方標數字
for i, row in enumerate(wing_stats.itertuples()):
    ax.text(i, row.attack_rate_pct + 1, f"{row.attack_rate_pct}%",
            ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## 4) 嚴重度 × 共病數熱力圖（seaborn）

共病越多的人是否更容易重症？用熱力圖交叉比較。

In [ ]:
severity_order = ["mild", "moderate", "severe"]
heat_data = (
    cases[cases["clinical_severity"].isin(severity_order)]
    .groupby(["clinical_severity", "n_comorbidities"])
    .size()
    .unstack(fill_value=0)
    .reindex(severity_order)
)

fig, ax = plt.subplots(figsize=(8, 3.5))
sns.heatmap(heat_data, annot=True, fmt="d", cmap="YlOrRd", ax=ax)
ax.set_title("臨床嚴重度 × 共病數")
ax.set_xlabel("共病數")
ax.set_ylabel("嚴重度")
plt.tight_layout()
plt.show()

## 5) 互動式分層流行曲線（Plotly）

用 Plotly 把流行曲線按樓層上色，滑鼠懸停可看數值。Plotly 的互動式圖表同樣需要遵循 CDC 流行曲線繪製規範：無間隙（`bargap=0`）、描述性標題、隱藏格線、Y 軸從 0 開始。

觀察：三個樓層的流行高峰是否同步？如果不同步，代表什麼？

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# 依樓層分層，並補齊完整日期範圍
daily_floor = (
    cases.groupby(["symptom_onset_date", "floor"])
    .size()
    .rename("cases")
    .reset_index()
)
daily_floor["floor"] = daily_floor["floor"].astype(str) + "F"

# 補齊所有日期 × 樓層組合（含 0 例的天數）
all_dates = pd.date_range(
    cases["symptom_onset_date"].min() - pd.Timedelta(days=3),
    cases["symptom_onset_date"].max() + pd.Timedelta(days=1),
    freq="D",
)
all_floors = sorted(daily_floor["floor"].unique())
full_idx = pd.MultiIndex.from_product([all_dates, all_floors], names=["symptom_onset_date", "floor"])
daily_floor = (
    daily_floor.set_index(["symptom_onset_date", "floor"])
    .reindex(full_idx, fill_value=0)
    .reset_index()
)

fig = px.bar(
    daily_floor,
    x="symptom_onset_date", y="cases", color="floor",
    barmode="stack",
    color_discrete_sequence=["#2c7fb8", "#41ae76", "#fe9929"],
    title="松柏護理之家退伍軍人症流行曲線，依樓層與發病日，2026 年 1 月",
    labels={"symptom_onset_date": "發病日期（Date of Symptom Onset）",
            "cases": "病例數（Number of Cases）",
            "floor": "樓層"},
)

# CDC 風格：無間隙、隱藏格線、Y 軸從 0 開始
fig.update_layout(
    bargap=0,                              # 長條之間無間隙
    xaxis=dict(showgrid=False),            # 隱藏垂直格線
    yaxis=dict(showgrid=False, rangemode="tozero"),  # 隱藏水平格線、Y 軸從 0
    plot_bgcolor="white",                  # 白色背景
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.show()

## 小結

這堂課你學會了 5 種流病常用圖表：

| 圖表 | 觀察重點 |
|------|----------|
| 流行曲線 | 峰值時間、上升/下降速度 → 傳播模式 |
| 年齡分布 | 感染者是否集中在特定年齡層 |
| 翼區長條圖 | 哪些翼區侵襲率異常偏高 → 空間線索 |
| 嚴重度×共病 | 共病多的人是否更容易重症 |
| 互動分層曲線 | 各樓層的流行高峰是否同步 |

下一章（Ch03 描述性統計），我們會把這些觀察量化——計算 2×2 表、卡方檢定、風險比。